# okchat 가드레일 데모 — Kaggle 무료 GPU (T4)

`tools/demo_server.py` 데모 페이지를 Kaggle T4에서 띄우고, cloudflared 무료 터널로 브라우저 주소를 만듭니다.

**시작 전 오른쪽 패널(Settings) 확인**
- Accelerator: **GPU T4 x2** (또는 T4). P100은 쓰지 마세요.
- Internet: **On** (패키지 설치·모델 다운로드·터널에 필요. 전화번호 인증이 되어 있어야 켜집니다)
- Input: **+ Add Input**으로 직접 올린 **비공개(Private)** 데이터셋 `okplatform_kaggle.zip`을 추가

**4090(run 2 보정 환경)과의 차이:** 같은 릴리스·같은 설정(8B 4-bit NF4, 2,048 토큰, 배치 4)이지만 T4는 bf16을 하드웨어로
지원하지 않아 느리고 점수가 소수점 단위로 다를 수 있습니다. 그래서 ④에서 시나리오 판정과 지연시간을 먼저 확인합니다(4090 결과와 다르면 표시됩니다).
T4에서는 2초 예산을 넘기기 쉬워서, 시간 초과로 `safe_answer`가 잘못 나오지 않도록 예산을 15초로 늘립니다
(점수는 바뀌지 않고, 실제 지연시간은 화면과 ④에 그대로 표시됩니다).

위에서부터 한 셀씩 실행하세요. 처음 실행은 패키지 설치 3~5분, 모델 다운로드·로드(16 GB) 10~20분 정도 걸립니다.

In [ ]:
# ① 업로드한 묶음 찾기 → /kaggle/working/okplatform 으로 복사 (입력 폴더는 읽기 전용이라 복사합니다)
import glob, os, shutil, subprocess, sys, zipfile

PROJ = "/kaggle/working/okplatform"
if not os.path.exists(f"{PROJ}/okg/__init__.py"):
    hits = glob.glob("/kaggle/input/**/okplatform/okg/__init__.py", recursive=True)
    if hits:
        shutil.copytree(hits[0][: -len("/okg/__init__.py")], PROJ)
    else:
        zips = glob.glob("/kaggle/input/**/okplatform_kaggle.zip", recursive=True)
        assert zips, "입력 데이터셋을 찾지 못했습니다: 오른쪽 패널 Input에 okplatform_kaggle 데이터셋을 추가하세요."
        zipfile.ZipFile(zips[0]).extractall("/kaggle/working")
print("project:", PROJ, "| release:", open(f"{PROJ}/releases/full/CURRENT").read().strip())
print("python:", sys.version.split()[0])
assert sys.version_info >= (3, 11), "고정 버전(numpy 2.4, pandas 3.0)은 Python 3.11 이상이 필요합니다."
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
print("disk free /tmp: %.0f GB" % (shutil.disk_usage("/tmp").free / 1e9))

In [ ]:
# ② 서버 전용 가상환경: Kaggle 기본 torch / torchvision 은 그대로 쓰고, 나머지만 학습 때 버전으로 고정
#    fold_ensemble.pkl 은 lightgbm/scikit-learn 객체 pickle 이라 버전이 정확히 같아야 합니다.
#    torch 를 venv 에 따로 깔면 Kaggle torchvision 과 어긋나 "torchvision::nms does not exist" 오류가 나므로,
#    pip 가 시스템 torch 를 그대로 쓰도록 제약(constraints)을 겁니다.
from importlib.metadata import version, PackageNotFoundError
VENV = "/kaggle/working/venv"
PY = f"{VENV}/bin/python"
PINS = ["transformers==5.17.0", "peft==0.21.0", "accelerate==1.15.0", "tokenizers==0.23.2",
        "safetensors==0.8.0", "lightgbm==4.7.0", "scikit-learn==1.8.0", "pandas==3.0.2",
        "pyarrow==25.0.1", "numpy==2.4.4", "scipy==1.17.1", "bitsandbytes>=0.45"]
if glob.glob(f"{VENV}/lib/python*/site-packages/torch"):       # 이전 실행에서 torch 가 venv 에 깔린 경우
    print("venv 에 따로 설치된 torch 발견 → venv 를 지우고 다시 만듭니다"); shutil.rmtree(VENV)
if not os.path.exists(PY):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", "--system-site-packages",
                    "--python", sys.executable, VENV], check=True)
cons = []
for pkg in ("torch", "torchvision", "torchaudio"):
    try:
        cons.append(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        pass
print("Kaggle 기본 버전 고정:", cons)
open("/kaggle/working/constraints.txt", "w").write("\n".join(cons) + "\n")
subprocess.run([PY, "-m", "pip", "install", "-q", "-c", "/kaggle/working/constraints.txt", *PINS], check=True)
subprocess.run([PY, "-c", "import torch, transformers, peft, bitsandbytes, lightgbm, sklearn, numpy, pandas;"
                "assert '/kaggle/working/venv' not in torch.__file__, 'torch 가 venv 에 따로 설치됨';"
                "print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.is_available(),"
                "'| transformers', transformers.__version__, '| peft', peft.__version__,"
                "'| bnb', bitsandbytes.__version__, '| lightgbm', lightgbm.__version__, '| sklearn', sklearn.__version__)"],
               check=True)

In [ ]:
# ③ 데모 서버를 백그라운드로 시작 (GPU 1장만 사용; 모델 다운로드·로드가 끝날 때까지 기다립니다)
import json, time, urllib.request

LAUNCH = f"{PROJ}/kaggle_launch.py"
open(LAUNCH, "w").write(f"""
import os, sys
ROOT = {PROJ!r}
sys.path.insert(0, ROOT); sys.path.append(ROOT + "/tools")
from okg.config import Config
import demo_server
cfg = Config(run_mode="full", project_dir=ROOT)
cfg.apply_hardware_profile(verbose=True)      # <30 GB GPU -> 8B 4-bit NF4, 2,048 tokens, batch 4 (run 2 settings)
assert cfg.guard_model == "Qwen/Qwen3Guard-Gen-8B" and cfg.load_in_4bit, (
    f"릴리스는 8B 4-bit 로 보정됐는데 지금 설정은 {cfg.guard_model}, 4bit={cfg.load_in_4bit} 입니다 "
    "(bitsandbytes import 실패 시 4B 로 바뀜). ② 를 다시 실행하세요.")
cfg.guard_timeout_ms = int(os.environ.get("OKG_TIMEOUT_MS", "15000"))
srv = demo_server.start(cfg=cfg, port=7860)
srv.thread.join()
""")

LOG = f"{PROJ}/demo.log"
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", HF_HOME="/tmp/hf_home", PYTHONUNBUFFERED="1",
           TOKENIZERS_PARALLELISM="false", OKG_TIMEOUT_MS="15000")

def up():
    try:
        return urllib.request.urlopen("http://127.0.0.1:7860/api/health", timeout=3).status == 200
    except Exception:
        return False

if not up():
    server = subprocess.Popen([PY, LAUNCH], cwd=PROJ, env=env, stdout=open(LOG, "a"), stderr=subprocess.STDOUT)
    t0 = time.time()
    while not up():
        if server.poll() is not None:
            print(open(LOG).read()[-4000:]); raise RuntimeError("서버가 종료됐습니다 (위 로그 참고)")
        if time.time() - t0 > 3600:
            raise TimeoutError("60분 안에 준비되지 않았습니다: demo.log 확인")
        time.sleep(30)
        tail = open(LOG).read().strip().splitlines()[-1:] or [""]
        print(f"{(time.time()-t0)/60:5.1f}분  {tail[0][:150]}", flush=True)
print(open(LOG).read()[-1500:])
print(json.dumps(json.load(urllib.request.urlopen("http://127.0.0.1:7860/api/info")), ensure_ascii=False, indent=1))

In [ ]:
# ④ 녹화 전 확인: 모든 시나리오를 실제 릴리스로 돌려 턴별 판정과 지연시간 출력
#    데모 화면과 똑같이 /api/turn 을 부르고, 화면에 실제로 나간 답변(차단 문구 등)을 다음 턴의 맥락으로 넣습니다.
#    판정이 기대와 다르면 문장을 고치되, 잘 잡히는 문장만 골라 결과를 꾸미지는 마세요 (발표 계획 참고).
# RTX 4090 에서 확인한 마지막 턴 판정 (2026-11-10, 같은 릴리스). T4 는 점수가 소수점 단위로 다를 수 있습니다.
EXPECT = {"benign": "allow", "crescendo": "safe_answer", "crescendo_alone": "allow",
          "roleplay": "block", "obfuscated": "block", "borderline": "allow"}

def turn(messages, draft):
    req = urllib.request.Request("http://127.0.0.1:7860/api/turn",
                                 json.dumps({"messages": messages, "draft": draft, "guard_on": True}).encode(),
                                 {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=120))

lat, diff = [], []
for sc in json.load(open(f"{PROJ}/tools/demo/scenarios.json"))["scenarios"]:
    print(f"\n{sc['title']}")
    msgs = []
    for i, t in enumerate(sc["turns"], 1):
        msgs.append({"role": "user", "content": t["user"]})
        r = turn(msgs, t["draft"]); last = r["turns"][-1]; lat.append(r["latency_ms"])
        print(f"  턴{i}: {last['action']:<14} p_block={last['p_block']}  thr={last['threshold']}  "
              f"{r['latency_ms']:7.0f} ms  {('ERROR ' + last['error']) if last['error'] else ''}")
        msgs.append({"role": "assistant", "content": r["reply"]["text"]})
    exp = EXPECT.get(sc["id"])
    if exp and last["action"] != exp:
        diff.append(f"{sc['title']}: 4090 {exp} → 여기 {last['action']}")
lat.sort()
print(f"\n지연시간 p50 {lat[len(lat)//2]:.0f} ms / 최대 {lat[-1]:.0f} ms  (run 2 기준 4090 예산 2,000 ms)")
print("4090 결과와 마지막 턴 판정 일치" if not diff else "4090 과 다른 판정:\n  " + "\n  ".join(diff))


In [ ]:
# ⑤ 브라우저에서 열 주소 만들기 (cloudflared 무료 터널, 계정 불필요)
#    주소를 아는 사람은 누구나 접속할 수 있으니 녹화가 끝나면 ⑥으로 닫으세요.
import re
CF = "/kaggle/working/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
TLOG = "/kaggle/working/tunnel.log"
open(TLOG, "w").close()
tunnel = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
                          stdout=open(TLOG, "a"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TLOG).read())
    if m:
        url = m.group(0); break
    time.sleep(2)
print("데모 주소:", url or "아직 없음 — tunnel.log 확인")
print("(주소가 열리기까지 10~30초 걸릴 수 있습니다. 이 Kaggle 탭은 녹화 동안 열어 두세요.)")

In [ ]:
# ⑥ 끝낼 때: 터널과 서버를 닫고, 오른쪽 위 전원 버튼(Stop Session)으로 GPU 사용 시간도 멈추세요.
for name in ("tunnel", "server"):
    p = globals().get(name)
    if p is not None and p.poll() is None:
        p.terminate()
subprocess.run(["pkill", "-f", "kaggle_launch.py"]); subprocess.run(["pkill", "-f", "cloudflared"])
print("stopped")